# 01 · Colab A100 80GB 启动 ComfyUI WebUI（按独立项目挂载云端资产）

- 填写 `PROJECT_ID`（例如测试短片 `01_rainy_signal`），ComfyUI 会自动将 `images/`、`characters/<character_id>/`、`outputs/`、`workflows/` 软链接直连到 `/MyDrive/AI_Video/projects/<PROJECT_ID>/`，所有素材、人物形象锚定图、专属工作流和生成视频全部隔离保存在该项目云端文件夹下。
- **Drive API Turbo 极速预热**：默认绕过 DriveFS FUSE，通过 Google Drive v3 API 多线程 Range 分片直下模型到本地 NVMe SSD，大幅缩短 A100 启动耗时且消除 ~50GB DriveFS 缓存膨胀。
- 调完工作流导出 `API Format JSON` 后，请及时 **断开并删除运行时** 省 A100 计算单元。

In [ ]:
#@title 1. 挂载 5TB Google Drive、开启 Drive API Turbo 授权、拉取代码并检查 GPU
REPO_URL = "https://github.com/WSQ233/AI-Video.git" #@param {type:"string"}
REPO_DIR = "/content/AI-Video" #@param {type:"string"}
USE_DRIVE_API_TURBO = True #@param {type:"boolean"}

!nvidia-smi

from google.colab import drive
drive.mount('/content/drive')

if USE_DRIVE_API_TURBO:
    from google.colab import auth
    auth.authenticate_user()
    print('[Drive API Turbo] Google OAuth 授权完成，A100 模型预热将绕过 DriveFS 多线程直下！')

import os, subprocess
if os.path.exists(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)


In [ ]:
#@title 2. 选择独立项目并从云端共享库极速预热模型包到本地 SSD
PROJECT_ID = "01_rainy_signal" #@param {type:"string"}
ACTIVE_PACKS = "minimax_h3_fl2va_bf16 minimax_h3_turbo_lora" #@param {type:"string"}
MODEL_LOAD_MODE = "copy_to_local" #@param ["copy_to_local", "symlink"]
STAGE_BACKEND = "auto" #@param ["auto", "drive_api", "drivefs"]
RANGE_WORKERS = 16 #@param {type:"integer"}
INSTALL_SAGEATTENTION = False #@param {type:"boolean"}
ENABLE_LTX_NODE = False #@param {type:"boolean"}

import os, subprocess

sub_env = os.environ.copy()
if STAGE_BACKEND != 'drivefs':
    try:
        import google.auth
        from google.auth.transport.requests import Request
        creds, _ = google.auth.default(scopes=['https://www.googleapis.com/auth/drive'])
        creds.refresh(Request())
        if getattr(creds, 'token', None):
            sub_env['GDRIVE_ACCESS_TOKEN'] = creds.token
    except Exception as exc:
        print(f'[提示] 未预刷新 Drive API Token ({exc})，将尝试 ADC 或回退 DriveFS。')

cmd = [
    'python', '-u', f'{REPO_DIR}/scripts/setup_env.py',
    '--repo-root', REPO_DIR,
    '--project', PROJECT_ID,
    '--mode', MODEL_LOAD_MODE,
    '--stage-backend', STAGE_BACKEND,
    '--range-workers', str(RANGE_WORKERS),
    '--packs', *ACTIVE_PACKS.split(),
]
if INSTALL_SAGEATTENTION:
    cmd.append('--sageattention')
if ENABLE_LTX_NODE:
    cmd.append('--enable-ltx')

subprocess.run(cmd, check=True, env=sub_env)


In [ ]:
#@title 3. 启动 ComfyUI 并通过 Cloudflare Tunnel 打开 WebUI
import json, re, shlex, subprocess, threading, time
from pathlib import Path

if not Path('/usr/local/bin/cloudflared').exists():
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
    !chmod +x /usr/local/bin/cloudflared

launch_info = json.loads(Path('/content/ComfyUI/launch_info.json').read_text(encoding='utf-8'))
comfy_dir = launch_info['comfyui_dir']
comfy_args = shlex.split(launch_info['comfyui_args'])
print(f"当前项目: {launch_info['project_id']} | 云端目录: {launch_info['project_dir']}")
print(f"GPU 档位: {launch_info['gpu_tier']} | 启动参数: {launch_info['comfyui_args']}")

def start_tunnel():
    time.sleep(6)
    p = subprocess.Popen(
        ['cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8188'],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    for line in p.stdout:
        m = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
        if m:
            print('\n' + '=' * 64)
            print('ComfyUI WebUI 访问地址:', m.group(0))
            print('=' * 64 + '\n', flush=True)

threading.Thread(target=start_tunnel, daemon=True).start()
subprocess.run(['python', 'main.py', *comfy_args], cwd=comfy_dir, check=True)


In [ ]:
#@title 4. (可选) 调试稳定后冻结所有节点 Commit SHA 备份到云端 shared/logs/
!python {REPO_DIR}/scripts/freeze_locks.py --comfy-dir /content/ComfyUI --lock-file {REPO_DIR}/configs/nodes_lock.json --out /content/drive/MyDrive/AI_Video/shared/logs/nodes_lock_frozen.json
